In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_splitfit_transform
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# 1. 데이터 준비
# make_classification: n_samples(샘플 수), n_features(특성 수)로 가상의 분류용 데이터셋을 생성합니다.
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)

# train_test_split: 전체 데이터를 학습용과 테스트용으로 8:2(test_size=0.2) 비율로 분리합니다.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. 개별 모델 학습 및 예측 (확률값 추출)
# fit: 주어진 학습 데이터(X_train, y_train)로 모델을 학습시킵니다.
m1 = RandomForestClassifier(random_state=42).fit(X_train, y_train)
m2 = GradientBoostingClassifier(random_state=42).fit(X_train, y_train)
m3 = LogisticRegression(random_state=42).fit(X_train, y_train)

# predict_proba: 각 클래스(0과 1)에 속할 예측 '확률'을 반환합니다. [:, 1]은 양성(1) 클래스의 확률만 가져옵니다.
p1 = m1.predict_proba(X_test)[:, 1]
p2 = m2.predict_proba(X_test)[:, 1]
p3 = m3.predict_proba(X_test)[:, 1]

# --- (1) 단순 평균 앙상블 ---
# np.mean: 여러 배열을 합쳐서 평균을 구합니다. axis=0은 같은 인덱스(행)끼리 평균을 계산함을 의미합니다.
avg_pred = np.mean([p1, p2, p3], axis=0)
avg_label = (avg_pred >= 0.5).astype(int)

# accuracy_score: 실제 정답(y_test)과 예측 정답(avg_label)을 비교하여 정확도(비율)를 구합니다.
print(f"단순 평균 앙상블 정확도: {accuracy_score(y_test, avg_label):.4f}")

# --- (2) 가중치 평균 앙상블 ---
weights = [0.2, 0.5, 0.3]
# np.average: np.mean과 유사하지만, weights 파라미터를 통해 각 배열에 가중치를 곱해 가중 평균을 반영합니다.
w_avg_pred = np.average([p1, p2, p3], axis=0, weights=weights)
w_avg_label = (w_avg_pred >= 0.5).astype(int)
print(f"가중치(0.2, 0.5, 0.3) 앙상블 정확도: {accuracy_score(y_test, w_avg_label):.4f}")

# --- (3) 최적 가중치 조합 탐색 ---
best_score = 0
best_weights = None

# np.arange(시작, 끝, 간격): 0부터 1.1 전까지 0.1 간격으로 [0.0, 0.1, ... 1.0] 배열을 만듭니다.
for w1 in np.arange(0, 1.1, 0.1):
    for w2 in np.arange(0, 1.1 - w1, 0.1):
        w3 = 1.0 - w1 - w2
        if w3 < -1e-5: continue 
        
        pred = w1*p1 + w2*p2 + w3*p3
        score = accuracy_score(y_test, (pred >= 0.5).astype(int))
        
        if score > best_score:
            best_score = score
            best_weights = (np.round(w1, 1), np.round(w2, 1), np.round(w3, 1))

print(f"\n최적 가중치 조합 탐색 결과 (M1, M2, M3): {best_weights}")
print(f"최적 가중치 앙상블 정확도: {best_score:.4f}")

단순 평균 앙상블 정확도: 0.8900
가중치(0.2, 0.5, 0.3) 앙상블 정확도: 0.8900

최적 가중치 조합 탐색 결과 (M1, M2, M3): (np.float64(0.0), np.float64(1.0), np.float64(0.0))
최적 가중치 앙상블 정확도: 0.9100
